In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

from rapidfuzz import fuzz, process
import ftfy

from zipfile import ZipFile
from urllib.request import urlopen
from io import BytesIO
import unicodedata

import joblib

In [6]:
us_state_to_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands, U.S.": "VI",
} 

abbrev_to_us_state = dict(map(reversed, us_state_to_abbrev.items()))
# https://gist.github.com/rogerallen/1583593

In [3]:
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vTZTum7NEaS6hvniyNJbDTWLDY0rATtgR5yXV2EUbtt6ubRPqv1sVH3PdJM3rgAzS_ak_XSgxLtaq3-/pub?output=csv'
candinfo = pd.read_csv(url)
candinfo.to_csv('../../2026_data/2026_senate_candidateinfo.csv')
candinfo.head()

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any
0,Alabama,Everett Weiss,Barry Moore,False,False
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True
3,Colorado,John Hickenlooper,Mark Baisley,True,False
4,Delaware,Chris Coons,Michael Katz,True,False


In [4]:
for party in ['dem', 'rep']:
    candinfo[f'{party}_cand'] = candinfo[f'{party}_cand'].fillna('TBD').astype(str)
    candinfo[f'{party}_inc_any'] = candinfo[f'{party}_inc_any'].astype(bool)

In [5]:
dem_only_candinfo = candinfo[candinfo['rep_cand'] == 'Not Contested']
rep_only_candinfo = candinfo[candinfo['dem_cand'] == 'Not Contested']
candinfo = candinfo[(candinfo['dem_cand'] != 'Not Contested') &
    (candinfo['rep_cand'] != 'Not Contested')]

In [7]:
candinfo['state_po'] = candinfo['state'].map(us_state_to_abbrev)
candinfo.head()

,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po
0,Alabama,Everett Weiss,Barry Moore,False,False,AL
1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,AK
2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,AR
3,Colorado,John Hickenlooper,Mark Baisley,True,False,CO
4,Delaware,Chris Coons,Michael Katz,True,False,DE


In [8]:
candinfo.shape

(35, 6)

In [9]:
fec_webl_colnames = ["CAND_ID", "CAND_NAME", "CAND_ICI", "PTY_CD", "CAND_PTY_AFFILIATION", "TTL_RECEIPTS", "TRANS_FROM_AUTH", "TTL_DISB", "TRANS_TO_AUTH", "COH_BOP", "COH_COP", "CAND_CONTRIB", "CAND_LOANS", "OTHER_LOANS", "CAND_LOAN_REPAY", "OTHER_LOAN_REPAY", "DEBTS_OWED_BY", "TTL_INDIV_CONTRIB", "CAND_OFFICE_ST", "CAND_OFFICE_DISTRICT", "SPEC_ELECTION", "PRIM_ELECTION", "RUN_ELECTION", "GEN_ELECTION", "GEN_ELECTION_PRECENT", "OTHER_POL_CMTE_CONTRIB", "POL_PTY_CONTRIB", "CVG_END_DT", "INDIV_REFUNDS", "CMTE_REFUNDS"]

In [10]:
fec_cn_colnames = ["CAND_ID", "CAND_NAME", "CAND_PTY_AFFILIATION", "CAND_ELECTION_YR", "CAND_OFFICE_ST", "CAND_OFFICE", "CAND_OFFICE_DISTRICT", "CAND_ICI", "CAND_STATUS", "CAND_PCC", "CAND_ST1", "CAND_ST2", "CAND_CITY", "CAND_ST", "CAND_ZIP"]

In [11]:
webl = pd.read_table('../../2026_data/fec/webl26.txt', sep="|", names=fec_webl_colnames)
cn = pd.read_table('../../2026_data/fec/cn.txt', sep="|", names=fec_cn_colnames)
fec = pd.merge(left=webl, right=cn, on='CAND_ID', how='left')
fec = fec[[col for col in fec.columns.values if ('_y' not in col)]]
fec.columns = fec.columns.str.strip('_x')
fec.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,CMTE_REFUNDS,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,H2AK01158,"PELTOLA, MARY",C,1,DEM,152304.86,0.00,232791.29,0.0,83969.49,...,0.00,2026.0,H,C,C00812388,810 N STREET,SUITE 301,ANCHORAGE,AK,99501.0
1,H2AK01083,"BEGICH, NICHOLAS III",I,2,REP,5470446.24,1601616.78,2455618.51,0.0,104330.06,...,4382.86,2026.0,H,C,C00792341,PO BOX 671710,NaN,CHUGIAK,AK,99567.0
2,H6AK01092,"HILL, BILL",C,3,IND,1336291.63,7400.00,820405.67,0.0,0.00,...,0.00,2026.0,H,C,C00935437,PO BOX 220703,NaN,ANCHORAGE,AK,99522.0
3,H6AL01094,"JONES, CLYDE W MR. JR",O,1,DEM,85967.63,0.00,57937.52,0.0,0.00,...,0.00,2026.0,H,C,C00920918,11637 WENTWOOD CT,NaN,DAPHNE,AL,36526.0
4,H0AL01055,"CARL, JERRY LEE, JR",O,2,REP,1162582.80,183453.86,865256.75,0.0,69290.42,...,1000.00,2026.0,H,C,C00697789,PO BOX 852138,NaN,MOBILE,AL,36685.0


In [12]:
fec = fec[fec['CAND_OFFICE'] == 'S']
fec['CAND_OFFICE_ST'] = fec['CAND_OFFICE_ST'].astype(str)
fec['CAND_NAME'] = fec['CAND_NAME'].astype(str)
fec['CAND_PTY_AFFILIATION'] = fec['CAND_PTY_AFFILIATION'].replace({'DFL': 'DEM'})

In [13]:
fec.columns.values

array(['CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD',
       'CAND_PTY_AFFILIATION', 'TTL_RECEIPTS', 'TRANS_FROM_AUTH',
       'TTL_DISB', 'TRANS_TO_AUTH', 'COH_BOP', 'COH_COP', 'CAND_CONTRIB',
       'CAND_LOANS', 'OTHER_LOANS', 'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY',
       'DEBTS_OWED_BY', 'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST',
       'CAND_OFFICE_DISTRICT', 'SPEC_ELECTION', 'PRIM_ELECTION',
       'RUN_ELECTION', 'GEN_ELECTION', 'GEN_ELECTION_PRECENT',
       'OTHER_POL_CMTE_CONTRIB', 'POL_PTY_CONTRIB', 'CVG_END_DT',
       'INDIV_REFUNDS', 'CMTE_REFUNDS', 'CAND_ELECTION_YR', 'CAND_OFFICE',
       'CAND_STATUS', 'CAND_PCC', 'CAND_ST1', 'CAND_ST2', 'CAND_CITY',
       'CAND_ST', 'CAND_ZIP'], dtype=object)

In [15]:
fec.shape

(336, 39)

In [16]:
def get_fuzzymatch_cand(state_po, party, candidate):
    error_return = 'no_match'
    if candidate == 'TBD':
        return error_return
    
    df = fec[(fec['CAND_OFFICE_ST'] == state_po) &
        (fec['CAND_PTY_AFFILIATION'].isin([party, 'IND', 'NON', 'OTH']))]

    # df['cand_name_lst'] = df['CAND_NAME'].str.split(',')
    # df['first_name_cand'] = df['cand_name_lst'].map(lambda x: x[1])
    # df['last_name_cand'] = df['cand_name_lst'].map(lambda x: x[0])
    # df['cand_name_ordered'] = df['first_name_cand'] + ' ' + df['last_name_cand']

    if df.shape[0] == 0:
        return error_return

    if '/' in candidate:
        cands = candidate.split(separator='/')
        matches, contribs = [], []
        for c in cands:
            fuzzymatch = process.extractOne(c, df['CAND_NAME'].values, scorer=fuzz.WRatio, score_cutoff=55)
            indiv_contrib = df[df['CAND_NAME'] == fuzzymatch]['TTL_INDIV_CONTRIB'].values[0]
            if fuzzymatch is not None:
                matches.append(fuzzymatch[0])
                contribs.append(indiv_contrib)
            else:
                matches.append(error_return)
                contribs.append(0)
        return matches

    else:
        fuzzymatch = process.extractOne(candidate, df['CAND_NAME'].values, scorer=fuzz.token_sort_ratio, score_cutoff=10)
        if fuzzymatch is not None:
            indiv_contrib = df[df['CAND_NAME'] == fuzzymatch[0]]['TTL_INDIV_CONTRIB'].values[0]
        else:
            indiv_contrib = 0
        return error_return if fuzzymatch is None else fuzzymatch[0]